In [ ]:
!pip install mlxtend

In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
t_and_a_df_ASC = pd.read_pickle("t&a_ASC_association_rules.pk1")
t_and_a_df_MAIN = pd.read_pickle("t&a_MAIN_association_rules.pk1")

In [3]:
"""
1) CLEAN + DISCRETIZE NUMERIC FIELDS
"""
def clean_discretize_numeric_fields(df):
    fixed_bins = {
        'FIRST_PACU_PAIN_SCORE': [0,2,4,6,8,10],
        'HIGHEST_PACU_PAIN_SCORE': [0,2,4,6,8,10],
        'age': [0,2,5,11,18],
        'length_of_stay_in_minutes': [0,90,200,1000],
        'ASA_SCORE_C':[0,1,2,3,4,5]
    }

    for col, bins in fixed_bins.items():
        df[col+"_grouped"] = pd.cut(df[col], bins=bins, right = True)

In [4]:
clean_discretize_numeric_fields(t_and_a_df_ASC)
clean_discretize_numeric_fields(t_and_a_df_MAIN)

In [5]:
#filter for columns needed
def filter_columns(df):
    new_df = df[['HIGHEST_PACU_PAIN_SCORE_grouped','FIRST_PACU_PAIN_SCORE_grouped','age_grouped','med_combos','length_of_stay_in_minutes_grouped', 'GENDER', 'RACE', 'ASA_SCORE_C_grouped']]
    return new_df

In [6]:
t_and_a_df_ASC_filtered = filter_columns(t_and_a_df_ASC)
t_and_a_df_MAIN_filtered = filter_columns(t_and_a_df_MAIN)

In [7]:
"""
2) ONE-HOT ENCODE INTO BOOLEAN MATRIX
"""
def one_hot_encode(df):
    onehot_tonsillectomy = pd.get_dummies(df, prefix_sep="=", dtype=bool)
    return onehot_tonsillectomy

In [8]:
t_and_a_df_ASC_onehot = one_hot_encode(t_and_a_df_ASC_filtered)
t_and_a_df_MAIN_onehot = one_hot_encode(t_and_a_df_MAIN_filtered)

In [9]:
"""
3) APRIORI - FREQUENT ITEMSETS
"""
def create_apriori(df):
    tonsillectomy_itemsets = apriori(df, min_support=0.005, use_colnames=True, max_len=2).sort_values('support', ascending=False)
    return tonsillectomy_itemsets

In [10]:
t_and_a_df_ASC_onehot.dropna(inplace = True)
t_and_a_df_MAIN_onehot.dropna(inplace = True)

t_and_a_df_ASC_itemset = create_apriori(t_and_a_df_ASC_onehot)
t_and_a_df_MAIN_itemset = create_apriori(t_and_a_df_MAIN_onehot)

In [11]:
"""
4) ASSOCIATION RULES
"""
def fs2str(fs):
    return ", ".join(sorted(map(str,fs)))

t_and_a_df_ASC_rules = association_rules(
    t_and_a_df_ASC_itemset,
    metric='support',
    min_threshold=0.006,
).sort_values(['lift', 'confidence'], ascending=[False,False])

t_and_a_df_MAIN_rules = association_rules(
    t_and_a_df_MAIN_itemset,
    metric='support',
    min_threshold=0.006,
).sort_values(['lift', 'confidence'], ascending=[False,False])

t_and_a_df_ASC_rules['antecedents_str'] = t_and_a_df_ASC_rules['antecedents'].apply(fs2str)
t_and_a_df_ASC_rules['consequents_str'] = t_and_a_df_ASC_rules['consequents'].apply(fs2str)

t_and_a_df_MAIN_rules['antecedents_str'] = t_and_a_df_MAIN_rules['antecedents'].apply(fs2str)
t_and_a_df_MAIN_rules['consequents_str'] = t_and_a_df_MAIN_rules['consequents'].apply(fs2str)

In [12]:
filtered_ASC_rules = t_and_a_df_ASC_rules.query('(lift > 1.0 or lift < 0.9) and confidence > 0 and support >= 0')

PACU Pain Score vs. Medication Combinations

In [ ]:
filtered_ASC_rules[
    filtered_ASC_rules['consequents'].apply(
        lambda x: any('HIGHEST_PACU_PAIN_SCORE_grouped' in item for item in x)
    )
].sort_values(by='antecedents',
              key = lambda col: col.astype(str))

Obtaining Baseline Pain Score Values

In [10]:
patient_pain_score_count = {}
for pain_score in t_and_a_patient_data_all["HIGHEST_PACU_PAIN_SCORE_grouped"].unique():
    patient_pain_score_count[pain_score] = len(t_and_a_patient_data_all[t_and_a_patient_data_all["HIGHEST_PACU_PAIN_SCORE_grouped"] == pain_score])/len(t_and_a_patient_data_all)
patient_pain_score_count

{Interval(4.0, 6.0, closed='right'): 0.33378248315688164,
 Interval(0.0, 2.0, closed='right'): 0.0410009624639076,
 Interval(6.0, 8.0, closed='right'): 0.16342637151106834,
 Interval(2.0, 4.0, closed='right'): 0.12223291626564003,
 Interval(8.0, 10.0, closed='right'): 0.17324350336862368,
 nan: 0.0}

In [11]:
t_and_a_patient_data_all["HIGHEST_PACU_PAIN_SCORE_grouped"].unique()

[(4.0, 6.0], (0.0, 2.0], (6.0, 8.0], (2.0, 4.0], (8.0, 10.0], NaN]
Categories (5, interval[int64, right]): [(0, 2] < (2, 4] < (4, 6] < (6, 8] < (8, 10]]

**Calculating Relative Risk**

In [ ]:
!pip install statsmodels

In [14]:
import numpy as np

In [17]:
def create_RR_table(df, med_combo, interval, pain_type):
    pts_given_mor = df[df["med_combos"] == med_combo]
    exposed_yes = len(pts_given_mor[pts_given_mor[pain_type] == interval])
    exposed_no = len(pts_given_mor[pts_given_mor[pain_type] != interval])

    pts_not_given_mor = df[df["med_combos"] != med_combo]
    unexposed_yes = len(pts_not_given_mor[pts_not_given_mor[pain_type] == interval])
    unexposed_no = len(pts_not_given_mor[pts_not_given_mor[pain_type] != interval])

    table = np.array([
    [exposed_yes, exposed_no],   # Exposed: outcome yes, no
    [unexposed_yes, unexposed_no]   # Not exposed: outcome yes, no
    ])

    print(exposed_yes)
    print(unexposed_yes)

    return table

In [18]:
from statsmodels.stats.contingency_tables import Table2x2
def calculate_RR(table):
    t = Table2x2(table)

    rr = t.riskratio
    ci_low, ci_high = t.riskratio_confint()

    rr_rounded = round(rr, 2)
    ci_low_r = round(ci_low, 2)
    ci_high_r = round(ci_high, 2)
    return "Relative Risk: " + str(rr_rounded) + ", CI-low: " + str(ci_low_r) + ", CI-high: " + str(ci_high_r)

In [19]:
#Mor vs. Fen

#Pain Score = (0,2] - Mor
table = create_RR_table(t_and_a_df_ASC, "Mor", pd.Interval(0, 2, closed="right"), "FIRST_PACU_PAIN_SCORE_grouped")
print("Pain Score = (0,2]; Medication Combination: Mor")
print(calculate_RR(table))
print(" ")

#Pain Score = (2, 4] - Mor
table = create_RR_table("Mor", pd.Interval(2, 4, closed="right"))
print("Pain Score = (2,4]; Medication Combination: Mor")
print(calculate_RR(table))
print(" ")

#Pain Score = (6, 8] - Fen
table = create_RR_table("Fen", pd.Interval(6, 8, closed="right"))
print("Pain Score = (6,8]; Medication Combination: Fen")
print(calculate_RR(table))
print(" ")

#Pain Score = (6, 8] - Mor
table = create_RR_table("Mor", pd.Interval(6, 8, closed="right"))
print("Pain Score = (6,8]; Medication Combination: Mor")
print(calculate_RR(table))
print(" ")

#Pain Score = (8, 10] - Mor
table = create_RR_table("Mor", pd.Interval(8, 10, closed="right"))
print("Pain Score = (8,10]; Medication Combination: Mor")
print(calculate_RR(table))
print(" ")

2
7
Pain Score = (0,2]; Medication Combination: Mor
Relative Risk: 1.02, CI-low: 0.21, CI-high: 4.9
 


In [43]:
#MorAce vs. FenAce

#Pain Score = (4, 6] - MorAce
table = create_RR_table("MorAce", pd.Interval(4, 6, closed="right"))
print("Pain Score = (4,6]; Medication Combination: MorAce")
print(calculate_RR(table))
print(" ")

#Pain Score = (4, 6] - FenAce
table = create_RR_table("FenAce", pd.Interval(4, 6, closed="right"))
print("Pain Score = (4,6]; Medication Combination: FenAce")
print(calculate_RR(table))
print(" ")

#Pain Score = (6, 8] - FenAce
table = create_RR_table("FenAce", pd.Interval(6, 8, closed="right"))
print("Pain Score = (6,8]; Medication Combination: FenAce")
print(calculate_RR(table))
print(" ")

#Pain Score = (8, 10] - FenAce
table = create_RR_table("FenAce", pd.Interval(8, 10, closed="right"))
print("Pain Score = (8,10]; Medication Combination: FenAce")
print(calculate_RR(table))
print(" ")

Pain Score = (4,6]; Medication Combination: MorAce
Relative Risk: 0.96, CI-low: 0.87, CI-high: 1.05
 
Pain Score = (4,6]; Medication Combination: FenAce
Relative Risk: 0.79, CI-low: 0.67, CI-high: 0.92
 
Pain Score = (6,8]; Medication Combination: FenAce
Relative Risk: 1.19, CI-low: 0.97, CI-high: 1.46
 
Pain Score = (8,10]; Medication Combination: FenAce
Relative Risk: 2.1, CI-low: 1.81, CI-high: 2.44
 


In [ ]:
table = create_RR_table("MorDex", pd.Interval(0, 2, closed="right"))
print("Pain Score = (0,2]; Medication Combination: MorDex")
print(calculate_RR(table))
print(" ")

table = create_RR_table("MorDex", pd.Interval(8, 10, closed="right"))
print("Pain Score = (8,10]; Medication Combination: MorDex")
print(calculate_RR(table))
print(" ")

In [ ]:
#Pain Score = (8, 10] - FenAce
table = create_RR_table("FenAce", pd.Interval(8, 10, closed="right"))

In [ ]:
#MorDexAce vs. FenDexAce

#Pain Score = (0, 2] - MorDexAce
table = create_RR_table("MorDexAce", pd.Interval(0, 2, closed="right"))
print("Pain Score = (0,2]; Medication Combination: MorDexAce")
print(calculate_RR(table))
print(" ")

#Pain Score = (4, 6] - MorDexAce
table = create_RR_table("MorDexAce", pd.Interval(4, 6, closed="right"))
print("Pain Score = (4,6]; Medication Combination: MorDexAce")
print(calculate_RR(table))
print(" ")

#Pain Score = (4, 6] - FenDexAce
table = create_RR_table("FenDexAce", pd.Interval(4, 6, closed="right"))
print("Pain Score = (4,6]; Medication Combination: FenDexAce")
print(calculate_RR(table))
print(" ")

#Pain Score = (6, 8] - MorDexAce
table = create_RR_table("MorDexAce", pd.Interval(6, 8, closed="right"))
print("Pain Score = (6,8]; Medication Combination: MorDexAce")
print(calculate_RR(table))
print(" ")

#Pain Score = (8, 10] - MorDexAce
table = create_RR_table("MorDexAce", pd.Interval(8, 10, closed="right"))
print("Pain Score = (8,10]; Medication Combination: MorDexAce")
print(calculate_RR(table))
print(" ")

#Pain Score = (8, 10] - FenDexAce
table = create_RR_table("FenDexAce", pd.Interval(8, 10, closed="right"))
print("Pain Score = (8,10]; Medication Combination: FenDexAce")
print(calculate_RR(table))
print(" ")

Prevalence of Medication Combinations for Each Pain Score

In [69]:
def calc_prevalence(table, interval):
    new_table = table[table["HIGHEST_PACU_PAIN_SCORE_grouped"] == interval]
    tot_affected = len(new_table)
    print("total affected: " + str(tot_affected))
    med_prevalence = new_table["med_combos"].value_counts().reset_index()
    med_prevalence["prevalence"] = (med_prevalence["count"] / tot_affected).round(3) * 100

    print(med_prevalence)



In [ ]:
#(0,2]
calc_prevalence(t_and_a_patient_data_all, pd.Interval(0, 2, closed="right"))

In [ ]:
#(2,4]
calc_prevalence(t_and_a_patient_data_all, pd.Interval(2, 4, closed="right"))

In [ ]:
#(4,6]
calc_prevalence(t_and_a_patient_data_all, pd.Interval(4, 6, closed="right"))

In [ ]:
#(6,8]
calc_prevalence(t_and_a_patient_data_all, pd.Interval(6, 8, closed="right"))

In [ ]:
#(8,10]
calc_prevalence(t_and_a_patient_data_all, pd.Interval(8, 10, closed="right"))

In [ ]:
t_and_a_patient_data_all["med_combos"].value_counts().reset_index()

PACU Pain Score vs. PACU LOS Association Rules

In [ ]:
filtered_tonsillectomy_rules[
    filtered_tonsillectomy_rules['antecedents'].apply(
        lambda x: any('HIGHEST_PACU_PAIN_SCORE_grouped' in item for item in x)
    )
    &
    filtered_tonsillectomy_rules['consequents'].apply(
        lambda x: any('length_of_stay_in_minutes_grouped' in item for item in x)
    )
]

In [ ]:
filtered_tonsillectomy_rules[
    filtered_tonsillectomy_rules['antecedents'].apply(
        lambda x: any('HIGHEST_PACU_PAIN_SCORE_grouped' in item for item in x)
    )
    &
    filtered_tonsillectomy_rules['consequents'].apply(
        lambda x: any('length_of_stay_in_minutes_grouped' in item for item in x)
    )
]

PACU Pain Score vs. Gender

In [ ]:
filtered_tonsillectomy_rules[
    filtered_tonsillectomy_rules['antecedents'].apply(
        lambda x: any('HIGHEST_PACU_PAIN_SCORE_grouped' in item for item in x)
    )
    &
    filtered_tonsillectomy_rules['consequents'].apply(
        lambda x: any('GENDER' in item for item in x)
    )
]

In [ ]:
filtered_tonsillectomy_rules[
    filtered_tonsillectomy_rules['antecedents'].apply(
        lambda x: any('HIGHEST_PACU_PAIN_SCORE_grouped' in item for item in x)
    )
    &
    filtered_tonsillectomy_rules['consequents'].apply(
        lambda x: any('age_grouped' in item for item in x)
        and
        any('GENDER' in item for item in x)
    )
]

Pain Score vs. Age

In [31]:
def create_RR_table_pain_age(pain_score, age):
    pts_age = t_and_a_patient_data_all[t_and_a_patient_data_all["age_grouped"] == age]
    tar_age_tar_pain = len(pts_age[pts_age["HIGHEST_PACU_PAIN_SCORE_grouped"] == pain_score])
    tar_age_not_pain = len(pts_age[pts_age["HIGHEST_PACU_PAIN_SCORE_grouped"] != pain_score])

    pts_not_age = t_and_a_patient_data_all[t_and_a_patient_data_all["age_grouped"] != age]
    not_age_tar_pain = len(pts_not_age[pts_not_age["HIGHEST_PACU_PAIN_SCORE_grouped"] == pain_score])
    not_age_not_pain = len(pts_not_age[pts_not_age["HIGHEST_PACU_PAIN_SCORE_grouped"] != pain_score])

    table = np.array([
    [tar_age_tar_pain, tar_age_not_pain],   # Exposed: outcome yes, no
    [not_age_tar_pain, not_age_not_pain]   # Not exposed: outcome yes, no
    ])

    print(tar_age_tar_pain)
    #print(not_age_tar_pain)

    return table

In [ ]:
table = create_RR_table_pain_age(pd.Interval(8, 10, closed="right"), pd.Interval(2, 5, closed="right"))
calculate_RR(table)

In [ ]:
create_RR_table_pain_age(pd.Interval(8, 10, closed="right"), pd.Interval(11, 18, closed="right"))

In [ ]:
table = create_RR_table_pain_age(pd.Interval(4, 6, closed="right"), pd.Interval(11, 18, closed="right"))
calculate_RR(table)

In [ ]:
table = create_RR_table_pain_age(pd.Interval(0, 2, closed="right"), pd.Interval(2, 5, closed="right"))
calculate_RR(table)

Pain Score vs. Medication Combo vs. Age

In [ ]:
filtered_tonsillectomy_rules[
    filtered_tonsillectomy_rules['antecedents'].apply(
        lambda x: any('med_combos' in item for item in x)
        and
        any('age_grouped' in item for item in x)

    )
    &
    filtered_tonsillectomy_rules['consequents'].apply(
        lambda x: any('HIGHEST_PACU_PAIN_SCORE_grouped' in item for item in x)
    )
]

In [59]:
#exposed: patients in x age group and received y med_combo
#yes: experienced target pain score

#unexposed: patients not in x age group and received y med_combo
# AND patients in x age group and did NOT receive y med_combo
# AND patients not in x age group and did not receive y med_combo
def create_RR_table_pain_age_med(pain_score, age, med_combos):
    age_filtered = t_and_a_patient_data_all[t_and_a_patient_data_all["age_grouped"] == age]
    exposed = age_filtered[age_filtered["med_combos"] == med_combos]

    exposed_yes = len(exposed[exposed["HIGHEST_PACU_PAIN_SCORE_grouped"] == pain_score])
    exposed_no = len(exposed[exposed["HIGHEST_PACU_PAIN_SCORE_grouped"] != pain_score])

    age_filtered_not = t_and_a_patient_data_all[t_and_a_patient_data_all["age_grouped"] != age]
    unexposed_1 = age_filtered_not[age_filtered_not["med_combos"] == med_combos] #not age, med combo
    unexposed_2 = age_filtered_not[age_filtered_not["med_combos"] != med_combos] #not age, not med combo
    unexposed_3 = age_filtered[age_filtered["med_combos"] != med_combos] #age, not med combo

    unexposed = pd.concat([unexposed_1, unexposed_2, unexposed_3], ignore_index = True)

    unexposed_yes = len(unexposed[unexposed["HIGHEST_PACU_PAIN_SCORE_grouped"] == pain_score])
    unexposed_no = len(unexposed[unexposed["HIGHEST_PACU_PAIN_SCORE_grouped"] != pain_score])

    table = np.array([
    [exposed_yes, exposed_no],   # Exposed: outcome yes, no
    [unexposed_yes, unexposed_no]   # Not exposed: outcome yes, no
    ])

    print(exposed_yes)
    #print(not_age_tar_pain)

    return table

In [ ]:
table = create_RR_table_pain_age_med(pd.Interval(4, 6, closed="right"), pd.Interval(5, 11, closed="right"), "FenAce")
calculate_RR(table)

In [ ]:
table = create_RR_table_pain_age_med(pd.Interval(8, 10, closed="right"), pd.Interval(2, 5, closed="right"), "MorDexAce")
calculate_RR(table)